# === SORU 1 ===========================
$$\hat{\beta} = (X^{T}X)^{-1}X^{T}y$$

In [29]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

df = pd.DataFrame({
    'X1': [1, 4, 9, 11, 3, 8, 5, 10, 2, 7, 6, 1],
    'X2': [9, 2, -8, -10, 6, -6, 0, -12, 4, -2, -4, 8],
    'X3': ['A', 'A', 'C', 'C', 'B', 'B', 'C', 'C', 'A', 'C', 'B', 'B'],
    'Y': [14, 16, 2, 0, 10, 6, 4, -8, 20, -6, 10, 12]
})

df_dummy = pd.get_dummies(df, columns=['X3'], drop_first=False) # dummy degisken (C referans)
X = sm.add_constant(df_dummy[['X1', 'X2', 'X3_A', 'X3_B']].astype(float)).to_numpy() # C yi at ve intercept ekle
y = df_dummy['Y'].to_numpy()

# Daha sonra aşagıda elle hesaplarımızı, hazır kütüphane sonucu ile karşılaştırmak için kullandım.
_model = sm.OLS(y, X).fit()

betas = np.linalg.inv(X.T @ X) @ X.T @ y # katsayılar
print(f"Manuel hesaplanmıs katsayılar: {betas}")
print(f"statsmodels.OLS().fit.params : {_model.params}")


Manuel hesaplanmıs katsayılar: [ 7.26544411 -1.28583137 -0.30242803 13.9136359   8.32322508]


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.854
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                     10.25
Date:                Sat, 14 Nov 2026   Prob (F-statistic):            0.00473
Time:                        17:22:56   Log-Likelihood:                -30.812
No. Observations:                  12   AIC:                             71.62
Df Residuals:                       7   BIC:                             74.05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------

/usr/lib/python3/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=12 observations were given.
  return hypotest_fun_in(*args, **kwds)


# === SORU 2 ===========================

| Source | SS | df | MS | F |
|:---:|:---:|:---:|:---:|:---:|
| Regression | SSR | k | MSR = SSR/k | F = MSR/MSE |
| Error | SSE | N − k − 1 | MSE = SSE/(N − k − 1) | — |
| Total | SST | N − 1 | — | — |

SSR = $\sum_{i=1}^{N} (\hat{y}_i - \bar{y})^2$ Regresyondan Kaynaklı Hata Kare Toplamları

SSE = $\sum_{i=1}^{N} (y_i - \hat{y}_i)^2$ Hata Kare Toplamları

SST = $\sum_{i=1}^{N} (y_i - \bar{y})^2$ Hedef değişkenin varyasyonu


In [48]:
N = 12 # gözlem sayısı
J = np.ones((N, N)) # tamamen 1' lerden oluşan nxn matrix
SST = np.dot(y, y) - (1/N) * y @ J @ y
print(f"SST: {SST}")
SSE = np.dot(y, y) - betas @ X.T @ y
print(f"SSE: {SSE}")
SSR = SST - SSE
print(f"SSR: {SSR}")
k = 4
p = k + 1
MSR = SSR / k
MSE = SSE / (N-p)

print(f"statsmodels.OLS().fit.ssr(residual): {_model.ssr}\n") # sum of squared residuals yukarıda hesapladığım SSE (sum of squared errors) ile aynı, sadece farklı notasyon

F = MSR / MSE
print(f"Manuel F-Score               : {F}")
print(f"statsmodels.OLS().fit.fvalue : {_model.fvalue}\n")

p_degeri = stats.f.sf(F, k, N-p) # F(4, 7)
print(f"Manuel p-degeri: {p_degeri}")
print(f"statsmodels.OLS().fit.pvalue: {_model.f_pvalue}\n")


SST: 818.6666666666666
SSE: 119.38858723491057
SSR: 699.278079431756
statsmodels.OLS().fit.ssr(residual): 119.38858723491443

Manuel F-Score               : 10.25003031988085
statsmodels.OLS().fit.fvalue : 10.250030319880462

Manuel p-degeri: 0.004725226707303517
statsmodels.OLS().fit.pvalue: 0.004725226707304029

